In [56]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

df = pd.DataFrame({
    "age": [
        21, 24, 29, 35, 42, 31, 27, 50,
        46, 23, 38, 33, 28, 55, 40, 26,
        44, 36, 30, 48
    ],

    "income": [
        2200, 2600, 3100, 4200, 5100,
        None, 2900, 6200, 5700, 2400,
        4500, 3900, None, 6800, 4800,
        2700, 5300, 4100, 3400, 5900
    ],

    "service_level": [
        "basic", "basic", "standard", "standard", "premium",
        "standard", None, "premium", "premium", "basic",
        "standard", "standard", "basic", "premium", "premium",
        "basic", "premium", "standard", "standard", "premium"
    ],

    "device": [
        "mobile", "desktop", "mobile", "tablet", "desktop",
        "mobile", "tablet", "desktop", "mobile", "mobile",
        "desktop", None, "tablet", "desktop", "mobile",
        "mobile", "tablet", "desktop", "mobile", "desktop"
    ],

    "subscribed": [
        0, 0, 1, 1, 1,
        1, 0, 1, 0, 0,
        1, 0, 0, 1, 1,
        0, 1, 0, 1, 1
    ]
})

In [57]:
df.head()

,age,income,service_level,device,subscribed
0,21,2200.0,basic,mobile,0
1,24,2600.0,basic,desktop,0
2,29,3100.0,standard,mobile,1
3,35,4200.0,standard,tablet,1
4,42,5100.0,premium,desktop,1


In [58]:
X = df.drop(columns=["subscribed"])
y = df["subscribed"]

In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [60]:
numeric_cols = [
    "age",
    "income"
]

categorical_cols = [
    "device"
]

ordinal_cols = [
    "service_level"
]

In [61]:
# 시멘틱 타입별 파이프라인 생성
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

numeric_transformer = Pipeline([
    (
        "impute",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_transformer = Pipeline([
    (
        "impute",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

ordinal_transformer = Pipeline([
    (
        "impute",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "ordinal",
        OrdinalEncoder(
            handle_unknown="error",
            categories=[["basic", "standard", "premium"]]
        )
    )
])

In [62]:
# 파이프라인 하나로 병합
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_cols
    ),
    (
        "cat",
        categorical_transformer,
        categorical_cols
    ),
    (
        "ord",
        ordinal_transformer,
        ordinal_cols
    )
])

In [63]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(max_iter=1000)
    )
])

In [64]:
model.fit(X_train, y_train)
model.score(X_train, y_train)

0.8125

In [65]:
model.score(X_test, y_test)

0.5

In [66]:
from sklearn import set_config

set_config(display="text")

import sklearn

print(sklearn.__version__)

1.9.0


In [67]:
model

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['device']),
     

In [68]:
model.named_steps["preprocess"]

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'income']),
                                ('cat',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['device']),
                                ('ord',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinal',
                                     

In [69]:
model.named_steps["preprocess"].named_transformers_

{'num': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scaler', StandardScaler())]),
 'cat': Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')),
                 ('onehot', OneHotEncoder(handle_unknown='ignore'))]),
 'ord': Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')),
                 ('ordinal',
                  OrdinalEncoder(categories=[['basic', 'standard',
                                              'premium']]))])}

In [70]:
model.named_steps["preprocess"].get_feature_names_out()

array(['num__age', 'num__income', 'cat__device_desktop',
       'cat__device_mobile', 'cat__device_tablet', 'ord__service_level'],
      dtype=object)

In [71]:
model.named_steps["preprocess"]["num"]["scaler"].mean_

array([  37.4375, 4593.75  ])

In [72]:
model.named_steps["preprocess"]["num"]["impute"].statistics_

array([  36.5, 4650. ])

In [75]:
model.named_steps["preprocess"]["cat"]["onehot"].categories_

[array(['desktop', 'mobile', 'tablet'], dtype=object)]

In [81]:
type(model.named_steps["preprocess"]["ord"]["ordinal"].categories_[0])

numpy.ndarray

In [82]:
model.named_steps["preprocess"]["ord"]["ordinal"].categories_

[array(['basic', 'standard', 'premium'], dtype=object)]

In [85]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
logistic = model.named_steps["model"]
logistic.coef_

array([[ 0.4713096 ,  0.19445259,  0.51446993, -0.17888061, -0.33536463,
         0.25321819]])

In [88]:
feature_names

array(['num__age', 'num__income', 'cat__device_desktop',
       'cat__device_mobile', 'cat__device_tablet', 'ord__service_level'],
      dtype=object)

In [86]:
logistic.coef_.shape

(1, 6)

In [91]:
coef_info = pd.Series(
    logistic.coef_[0],
    index=feature_names
)

coef_info.name = "coef"
coef_info



num__age               0.471310
num__income            0.194453
cat__device_desktop    0.514470
cat__device_mobile    -0.178881
cat__device_tablet    -0.335365
ord__service_level     0.253218
Name: coef, dtype: float64

In [92]:
preprocessor = model.named_steps["preprocess"]

X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_test_pre = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

X_test_pre

,num__age,num__income,cat__device_desktop,cat__device_mobile,cat__device_tablet,ord__service_level
0,-1.808383,-1.989429,0.0,1.0,0.0,0.0
17,-0.158148,-0.410352,1.0,0.0,0.0,1.0
15,-1.258304,-1.573883,0.0,1.0,0.0,0.0
1,-1.478336,-1.656992,1.0,0.0,0.0,0.0


In [97]:
x = X_test_pre.iloc[0]
print(type(x))
x

<class 'pandas.Series'>


num__age              -1.808383
num__income           -1.989429
cat__device_desktop    0.000000
cat__device_mobile     1.000000
cat__device_tablet     0.000000
ord__service_level     0.000000
Name: 0, dtype: float64

In [96]:
lr = model.named_steps["model"]

print(lr.coef_.shape)
lr.coef_

(1, 6)


array([[ 0.4713096 ,  0.19445259,  0.51446993, -0.17888061, -0.33536463,
         0.25321819]])

In [95]:
lr.intercept_

array([0.65017426])

In [103]:
w = lr.coef_[0]
z = x.to_numpy() @ w + lr.intercept_[0]
z

np.float64(-0.7678640539736504)

In [102]:
model.predict_proba(X_train)[0]

array([0.16791355, 0.83208645])

In [104]:
X_test

,age,income,service_level,device
0,21,2200.0,basic,mobile
17,36,4100.0,standard,desktop
15,26,2700.0,basic,mobile
1,24,2600.0,basic,desktop


In [105]:
model.decision_function(
    X_test.iloc[[0]]
)

array([-0.76786405])

In [106]:
import numpy as np

prob = 1 / (1 + np.exp(-z))

prob

np.float64(0.3169413352960755)

In [107]:
model.predict_proba(
    X_test.iloc[[0]]
)

array([[0.68305866, 0.31694134]])

In [108]:
model.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'preprocess', 'model', 'preprocess__n_jobs', 'preprocess__remainder', 'preprocess__sparse_threshold', 'preprocess__transformer_weights', 'preprocess__transformers', 'preprocess__verbose', 'preprocess__verbose_feature_names_out', 'preprocess__num', 'preprocess__cat', 'preprocess__ord', 'preprocess__num__memory', 'preprocess__num__steps', 'preprocess__num__transform_input', 'preprocess__num__verbose', 'preprocess__num__impute', 'preprocess__num__scaler', 'preprocess__num__impute__add_indicator', 'preprocess__num__impute__copy', 'preprocess__num__impute__fill_value', 'preprocess__num__impute__keep_empty_features', 'preprocess__num__impute__missing_values', 'preprocess__num__impute__strategy', 'preprocess__num__scaler__copy', 'preprocess__num__scaler__with_mean', 'preprocess__num__scaler__with_std', 'preprocess__cat__memory', 'preprocess__cat__steps', 'preprocess__cat__transform_input', 'preprocess__cat__verbose', 'preprocess__cat

In [113]:
model.get_params()["model__C"]

1.0

In [114]:
model.set_params(
    model__C=0.01
)

model.fit(X_train, y_train)

print(model.score(X_train, y_train))
print(model.score(X_test, y_test))

0.6875
0.0


In [115]:
model.set_params(
    preprocess__num__scaler="passthrough"
)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   'passthrough')]),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['device']),
        